# Superadditivity Analysis — PathMNIST, DomainNet, CIFAR-100

Reads experiment outputs directly from Google Drive and runs the full analysis pipeline:
- Factorial ANOVA + interaction bootstrap (core_factorial)
- Interaction surface I(α, β) fitting (pout_sweep)
- Visualization

## 1. Mount Drive & clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.isdir('/content/quantify_superadditivity'):
    !git clone https://github.com/angelafoua/quantify_superadditivity /content/quantify_superadditivity
%cd /content/quantify_superadditivity
!git checkout main && git pull origin main

In [ ]:
!pip install -q hydra-core==1.3.2 omegaconf==2.3.0 h5py networkx statsmodels seaborn tabulate tqdm scikit-learn pandas scipy
!pip install -q -e . --no-deps
!python scripts/validate_setup.py

In [ ]:
from pathlib import Path

# Point directly at the Drive folder — no download needed
DRIVE_OUT = Path('/content/drive/MyDrive/superadditivity_outputs')

# Also check Shared with me path in case the folder lives there
if not DRIVE_OUT.exists():
    DRIVE_OUT = Path('/content/drive/Shareddrives/superadditivity_outputs')
if not DRIVE_OUT.exists():
    # Shared-with-me folders appear under MyDrive once you add a shortcut,
    # or under this path on some Colab mounts:
    DRIVE_OUT = Path('/content/drive/MyDrive/superadditivity_outputs')
    print('NOTE: If the folder is shared with you (not owned), add a shortcut ')
    print('to your MyDrive first: Drive UI -> right-click folder -> "Add shortcut to Drive"')

assert DRIVE_OUT.exists(), f'{DRIVE_OUT} not found. Check the path or add a Drive shortcut.'

summaries = sorted(DRIVE_OUT.rglob('summary.json'))
print(f'Results root: {DRIVE_OUT}')
print(f'Found {len(summaries)} summary.json files')

## 2. Setup

In [ ]:
import logging
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

from superadditivity.analysis.aggregator import ResultAggregator
from superadditivity.analysis.anova import two_way_anova, functional_anova
from superadditivity.analysis.effect_sizes import bootstrap_ci
from superadditivity.analysis.interaction_analyzer import InteractionAnalyzer
from superadditivity.analysis.mixed_effects import fit_mixed_effects
from superadditivity.analysis.multiple_comparisons import pairwise_ttests
from superadditivity.analysis.statistical_report import StatisticalReport
from superadditivity.utils.io import ensure_dir, save_json, load_json

print('Modules loaded.')

## 3. Discover experiment folders by dataset

In [ ]:
def find_experiment_dirs(root: Path) -> dict:
    """Group top-level experiment folders by dataset and experiment type."""
    experiments = {'core_factorial': {}, 'pout_sweep': {}}
    
    for summary in sorted(root.rglob('summary.json')):
        with open(summary) as f:
            data = json.load(f)
        dataset = data.get('dataset', 'unknown')
        exp_name = data.get('experiment_name', '')
        
        if 'core_factorial' in exp_name:
            experiments['core_factorial'].setdefault(dataset, []).append(summary)
        elif 'pout_sweep' in exp_name:
            experiments['pout_sweep'].setdefault(dataset, []).append(summary)
    
    print('=== Core Factorial ===')
    for ds, files in experiments['core_factorial'].items():
        print(f'  {ds}: {len(files)} runs')
    print('\n=== Pout Sweep ===')
    for ds, files in experiments['pout_sweep'].items():
        print(f'  {ds}: {len(files)} runs')
    
    return experiments

experiments = find_experiment_dirs(DRIVE_OUT)

## 4. Core Factorial Analysis (2×2 ANOVA + Interaction Test)

For each dataset that has the 4 factorial cells (A/B/C/D), run:
- Two-way ANOVA (data_regime × network_regime)
- Permutation ANOVA
- Pairwise t-tests
- Bootstrap interaction test (I > 0)
- Mixed-effects model

In [ ]:
def run_factorial_for_dataset(dataset_name: str, summary_paths: list) -> dict:
    """Run the full factorial analysis for one dataset."""
    records = []
    for p in summary_paths:
        with open(p) as f:
            data = json.load(f)
        data['_run_dir'] = str(p.parent)
        records.append(data)
    
    df = pd.DataFrame(records)
    
    # Normalize data_regime: map dirichlet_semantic -> noniid for the 2x2 factorial
    regime_map = {'dirichlet_semantic': 'noniid', 'iid': 'iid'}
    df['data_regime'] = df['data_regime'].map(lambda x: regime_map.get(x, x))
    
    print(f'\n{"=" * 60}')
    print(f'FACTORIAL ANALYSIS: {dataset_name.upper()}')
    print(f'{"=" * 60}')
    print(f'Runs loaded: {len(df)}')
    print(f'Conditions: {df[["data_regime", "network_regime"]].drop_duplicates().values.tolist()}')
    
    results = {'dataset': dataset_name, 'n_runs': len(df)}
    
    metric_col = 'final_cka_cross_community'
    if metric_col not in df.columns:
        print(f'  WARNING: {metric_col} not in columns, skipping.')
        return results
    
    n_conditions = df[['data_regime', 'network_regime']].drop_duplicates().shape[0]
    if n_conditions < 4:
        print(f'  WARNING: Only {n_conditions} conditions found (need 4 for 2x2). Skipping ANOVA.')
        print(f'  Available: {df[["data_regime", "network_regime"]].drop_duplicates().values.tolist()}')
        return results
    
    fact_df = df[['data_regime', 'network_regime', metric_col]].copy()
    fact_df.columns = ['data_regime', 'network_regime', 'metric']
    for seed_col in ['run_seed', 'graph_seed']:
        if seed_col in df.columns:
            fact_df[seed_col] = df[seed_col]
    
    # --- Two-way ANOVA ---
    print('\n--- Two-Way ANOVA ---')
    anova_result = two_way_anova(fact_df)
    results['anova'] = anova_result
    print(anova_result['table'].to_string())
    print('\n  Partial eta-squared:')
    for source, eta in anova_result['partial_eta_sq'].items():
        print(f'    {source}: {eta:.6f}')
    
    # --- Permutation ANOVA ---
    print('\n--- Permutation ANOVA ---')
    func_anova = functional_anova(fact_df, n_permutations=10000, seed=42)
    results['perm_anova'] = func_anova
    print(f'  Interaction F={func_anova["observed_F"]:.4f}, '
          f'permutation p={func_anova["perm_p_value"]:.4g}')
    
    # --- Pairwise t-tests ---
    print('\n--- Pairwise t-tests ---')
    groups = {}
    for name, group in fact_df.groupby(['data_regime', 'network_regime']):
        key = f'{name[0]}_{name[1]}'
        groups[key] = group['metric'].values
    
    pairs = pairwise_ttests(groups)
    results['pairwise'] = pairs
    for pair_result in pairs:
        if isinstance(pair_result, dict):
            print(f'  {pair_result.get("pair", "?")}: '
                  f't={pair_result.get("t_stat", 0):.3f}, '
                  f'p={pair_result.get("p_value", 1):.4g}, '
                  f'd={pair_result.get("cohens_d", 0):.3f}')
    
    # --- Bootstrap Interaction Test ---
    print('\n--- Bootstrap Interaction Test (I > 0) ---')
    cell_keys = sorted(groups.keys())
    print(f'  Group keys: {cell_keys}')
    
    # Match cells: A=iid+ER, B=noniid+ER, C=iid+SBM, D=noniid+SBM
    def match_cell(keys, data_pat, net_pats):
        return [k for k in keys if data_pat in k.lower() and any(p in k.lower() for p in net_pats)]
    
    A_key = match_cell(cell_keys, 'iid', ['erdos', 'er_', 'er ', 'dense'])
    B_key = match_cell(cell_keys, 'noniid', ['erdos', 'er_', 'er ', 'dense'])
    C_key = match_cell(cell_keys, 'iid', ['sbm', 'community'])
    D_key = match_cell(cell_keys, 'noniid', ['sbm', 'community'])
    
    # Fallback: if keys are like "iid_erdos_renyi", match by exact substrings
    if not A_key:
        A_key = [k for k in cell_keys if k.startswith('iid') and 'sbm' not in k]
    if not B_key:
        B_key = [k for k in cell_keys if k.startswith('noniid') and 'sbm' not in k]
    if not C_key:
        C_key = [k for k in cell_keys if k.startswith('iid') and 'sbm' in k]
    if not D_key:
        D_key = [k for k in cell_keys if k.startswith('noniid') and 'sbm' in k]
    
    if A_key and B_key and C_key and D_key:
        print(f'  Cells: A={A_key[0]}, B={B_key[0]}, C={C_key[0]}, D={D_key[0]}')
        print(f'  Mean drift — A: {groups[A_key[0]].mean():.6f}, B: {groups[B_key[0]].mean():.6f}, '
              f'C: {groups[C_key[0]].mean():.6f}, D: {groups[D_key[0]].mean():.6f}')
        
        interaction = InteractionAnalyzer.bootstrap_interaction_test(
            groups[A_key[0]], groups[B_key[0]],
            groups[C_key[0]], groups[D_key[0]],
            n_bootstrap=10000, seed=42,
        )
        results['interaction'] = {
            k: v for k, v in interaction.items() if k != 'bootstrap_distribution'
        }
        print(f'  I = {interaction["I_observed"]:.6f}')
        print(f'  95% CI: [{interaction["I_ci_lower"]:.6f}, {interaction["I_ci_upper"]:.6f}]')
        print(f'  p-value (I > 0): {interaction["p_value"]:.4g}')
        print(f'  Superadditive: {interaction["p_value"] < 0.05}')
    else:
        print(f'  Could not identify all 4 cells.')
        print(f'  A={A_key}, B={B_key}, C={C_key}, D={D_key}')
        print(f'  Keys found: {cell_keys}')
    
    # --- Mixed Effects ---
    print('\n--- Mixed-Effects Model ---')
    try:
        me_result = fit_mixed_effects(fact_df)
        results['mixed_effects'] = me_result
        print(f'  {me_result["summary"][:500]}')
    except Exception as e:
        print(f'  Mixed effects failed: {e}')
    
    return results

In [ ]:
factorial_results = {}
for dataset, paths in experiments['core_factorial'].items():
    factorial_results[dataset] = run_factorial_for_dataset(dataset, paths)

## 5. Pout Sweep — Interaction Surface I(α, β)

For each dataset with a p_out sweep, compute the interaction term at each (α, p_out) grid point and fit the parametric surface.

In [ ]:
import re

def run_surface_for_dataset(dataset_name: str, summary_paths: list) -> dict:
    """Run interaction surface analysis for one dataset's pout sweep."""
    records = []
    for p in summary_paths:
        with open(p) as f:
            data = json.load(f)
        data['_run_dir'] = str(p.parent)
        
        if 'p_out' not in data:
            combined = ' '.join(part.name for part in p.parents)
            m = re.search(r'pout([0-9]+(?:\.[0-9]+)?)', combined)
            if m:
                data['p_out'] = float(m.group(1))
        
        if 'dirichlet_alpha' not in data and 'data_regime' in data:
            alpha_map = {
                'iid': 1000.0, 'mild_noniid': 1.0,
                'moderate_noniid': 0.3, 'severe_noniid': 0.05,
            }
            data['dirichlet_alpha'] = alpha_map.get(data['data_regime'])
        
        if 'data_regime' not in data:
            combined = ' '.join(part.name for part in p.parents)
            for regime in ('severe_noniid', 'moderate_noniid', 'mild_noniid', 'iid'):
                if regime in combined:
                    data['data_regime'] = regime
                    break
        
        records.append(data)
    
    df = pd.DataFrame(records)
    print(f'\n{"=" * 60}')
    print(f'SURFACE ANALYSIS: {dataset_name.upper()}')
    print(f'{"=" * 60}')
    print(f'Runs loaded: {len(df)}')
    
    results = {'dataset': dataset_name, 'n_runs': len(df)}
    
    sweep_data = InteractionAnalyzer.compute_sweep_interactions(df)
    
    if len(sweep_data['I_values']) == 0:
        print('  No sweep points computed.')
        return results
    
    print(f'  Grid points: {len(sweep_data["I_values"])}')
    print(f'  \u03b1 range: {sweep_data["alpha_values"].min():.3f} \u2013 {sweep_data["alpha_values"].max():.3f}')
    print(f'  \u03b2 (p_out) range: {sweep_data["beta_values"].min():.3f} \u2013 {sweep_data["beta_values"].max():.3f}')
    print(f'  I range: {sweep_data["I_values"].min():.6f} \u2013 {sweep_data["I_values"].max():.6f}')
    
    results['sweep'] = {
        'alpha': sweep_data['alpha_values'].tolist(),
        'beta': sweep_data['beta_values'].tolist(),
        'I': sweep_data['I_values'].tolist(),
    }
    
    surface_fit = InteractionAnalyzer.fit_interaction_surface(
        sweep_data['alpha_values'],
        sweep_data['beta_values'],
        sweep_data['I_values'],
    )
    results['surface_fit'] = surface_fit
    print(f'\n  Surface fit R\u00b2 = {surface_fit["r_squared"]:.4f}')
    print(f'  Fit params: {surface_fit["params"]}')
    
    return results

In [ ]:
surface_results = {}
for dataset, paths in experiments['pout_sweep'].items():
    surface_results[dataset] = run_surface_for_dataset(dataset, paths)

## 6. Visualization

In [ ]:
from superadditivity.visualization.figure_style import set_publication_style
set_publication_style()

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

### 6a. Factorial cell means (bar chart per dataset)

In [ ]:
for dataset, paths in experiments['core_factorial'].items():
    records = [json.load(open(p)) for p in paths]
    df = pd.DataFrame(records)
    
    # Normalize data_regime for consistent grouping
    regime_map = {'dirichlet_semantic': 'noniid', 'iid': 'iid'}
    df['data_regime'] = df['data_regime'].map(lambda x: regime_map.get(x, x))
    
    cell_means = df.groupby(['data_regime', 'network_regime'])['final_cka_cross_community'].agg(['mean', 'std'])
    print(f'\n{dataset.upper()} — Cell Means (CKA cross-community):')
    print(cell_means.to_string())
    
    fig, ax = plt.subplots(figsize=(6, 4))
    cell_means['mean'].unstack().plot(kind='bar', yerr=cell_means['std'].unstack(), ax=ax, capsize=4)
    ax.set_title(f'{dataset} — Factorial Cell Means')
    ax.set_ylabel('CKA (cross-community)')
    ax.set_xlabel('Data Regime')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

### 6b. Interaction surface heatmaps (pout sweep)

In [ ]:
for dataset, res in surface_results.items():
    if 'sweep' not in res:
        continue
    
    alpha = np.array(res['sweep']['alpha'])
    beta = np.array(res['sweep']['beta'])
    I_vals = np.array(res['sweep']['I'])
    
    try:
        from superadditivity.visualization.interaction_surface import plot_interaction_surface
        fig = plot_interaction_surface(alpha, beta, I_vals)
        fig.suptitle(f'{dataset} \u2014 I(\u03b1, \u03b2) Surface', fontsize=14)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f'  Surface plot failed for {dataset}: {e}')
        fig, ax = plt.subplots(figsize=(7, 5))
        sc = ax.scatter(alpha, beta, c=I_vals, cmap='RdBu_r', s=80, edgecolors='k')
        plt.colorbar(sc, label='I (interaction term)')
        ax.set_xlabel('\u03b1 (Dirichlet)')
        ax.set_ylabel('\u03b2 (p_out)')
        ax.set_title(f'{dataset} \u2014 I(\u03b1, \u03b2)')
        plt.tight_layout()
        plt.show()

### 6c. Drift metric comparison across datasets

In [ ]:
all_sweep = []
for dataset, paths in experiments['pout_sweep'].items():
    for p in paths:
        with open(p) as f:
            data = json.load(f)
        data['_dataset'] = dataset
        all_sweep.append(data)

sweep_df = pd.DataFrame(all_sweep)

if 'dirichlet_alpha' in sweep_df.columns and 'final_cka_cross_community' in sweep_df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    for ds, grp in sweep_df.groupby('_dataset'):
        means = grp.groupby('dirichlet_alpha')['final_cka_cross_community'].mean()
        ax.plot(means.index, means.values, 'o-', label=ds, markersize=6)
    ax.set_xlabel('Dirichlet \u03b1 (lower = more heterogeneous)')
    ax.set_ylabel('CKA (cross-community)')
    ax.set_title('Drift vs Data Heterogeneity \u2014 All Datasets')
    ax.legend()
    ax.set_xscale('log')
    plt.tight_layout()
    plt.show()

## 7. Summary Table

In [ ]:
print('\n' + '=' * 70)
print('SUMMARY OF RESULTS')
print('=' * 70)

for dataset, res in factorial_results.items():
    print(f'\n--- {dataset.upper()} (Factorial) ---')
    if 'interaction' in res:
        ix = res['interaction']
        print(f'  I = {ix["I_observed"]:.6f}  '
              f'CI=[{ix["I_ci_lower"]:.6f}, {ix["I_ci_upper"]:.6f}]  '
              f'p={ix["p_value"]:.4g}  '
              f'Superadditive={ix["p_value"] < 0.05}')

for dataset, res in surface_results.items():
    print(f'\n--- {dataset.upper()} (Surface) ---')
    if 'surface_fit' in res:
        sf = res['surface_fit']
        print(f'  R² = {sf["r_squared"]:.4f}')
        print(f'  Params: {sf["params"]}')
    if 'sweep' in res:
        I_arr = np.array(res['sweep']['I'])
        print(f'  I range: [{I_arr.min():.6f}, {I_arr.max():.6f}]')
        print(f'  I > 0 in {(I_arr > 0).sum()}/{len(I_arr)} grid points')